In [48]:
import pandas as pd
import math
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

In [31]:
df_anime = pd.read_csv('anime.csv')
df_ratings = pd.read_parquet('ratings.parquet')

In [32]:
df_ratings

,user_id,anime_id,rating
0,0,0,8
1,0,1,6
2,0,2,9
3,0,3,10
4,0,4,9
...,...,...,...
6144921,47142,352,8
6144922,47142,51,7
6144923,47142,54,7
6144924,47142,1784,9


In [33]:
df_ratings

,user_id,anime_id,rating
0,0,0,8
1,0,1,6
2,0,2,9
3,0,3,10
4,0,4,9
...,...,...,...
6144921,47142,352,8
6144922,47142,51,7
6144923,47142,54,7
6144924,47142,1784,9


In [34]:
print(df_ratings['rating'].min())
print(df_ratings['user_id'].min())
print(df_ratings['anime_id'].min())

1
0
0


In [35]:
NUM_USERS = df_ratings['user_id'].nunique(dropna=True)
NUM_ITEMS = df_ratings['anime_id'].nunique(dropna=True)
print(f'Hay {NUM_USERS} usuarios y {NUM_ITEMS} animes')

X = df_ratings[['user_id', 'anime_id']]
y = df_ratings['rating']

# Hacemos el split para el modelo de recomendación
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Filas de entrenamiento: {X_train.shape[0]:,}")
print(f"Filas de test: {X_test.shape[0]:,}")

Hay 47143 usuarios y 6532 animes
Filas de entrenamiento: 4,915,940
Filas de test: 1,228,986


In [36]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Entrenando en: {device}")
print(f"¿CUDA disponible?: {torch.cuda.is_available()}")

Entrenando en: cuda
¿CUDA disponible?: True


In [37]:
import torch
print(torch.__version__)

2.5.1+cu121


In [38]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Entrenando en: {device}") # Debería salir 'cuda'

Entrenando en: cuda


In [39]:

latent_dim = 5
epochs = 10

In [41]:
class GMFModel(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.item_embedding = nn.Embedding(num_items, latent_dim)
        self.fc = nn.Linear(latent_dim, 1)

    def forward(self, user_ids, item_ids):
        u_emb = self.user_embedding(user_ids)
        i_emb = self.item_embedding(item_ids)
        interact = torch.mul(u_emb, i_emb)
        return self.fc(interact).flatten()
# Instanciamos el modelo
GMF0 = GMFModel(NUM_USERS, NUM_ITEMS, latent_dim)
print(GMF0)

GMFModel(
  (user_embedding): Embedding(47143, 5)
  (item_embedding): Embedding(6532, 5)
  (fc): Linear(in_features=5, out_features=1, bias=True)
)


In [43]:
import os
os.makedirs('modelos_entrenados', exist_ok=True)

###  Simple Perceptron

In [ ]:


# Configuración de optimizador y pérdida
optimizer = torch.optim.Adam(GMF0.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

# Mover el modelo a la GPU
GMF0 = GMF0.to(device)

# Preparación de datos (Corregido el acceso a columnas y conversión a .values)
users_t = torch.tensor(X_train['user_id'].values, dtype=torch.long)
items_t = torch.tensor(X_train['anime_id'].values, dtype=torch.long)
ratings_t = torch.tensor(y_train.values, dtype=torch.float32)

dataset = TensorDataset(users_t, items_t, ratings_t)

# Bucle de entrenamiento
# Añade num_workers y pin_memory para acelerar la transferencia a la GPU
loader = DataLoader(
    dataset, 
    batch_size=2048, 
    shuffle=True, 
    num_workers=4,     # Usa subprocesos de la CPU para precargar datos
    pin_memory=True    # Agiliza la copia de memoria hacia la VRAM de la 3060 Ti
)

ruta = Path("GMF0_wights.pth")
if not ruta.exists():
    
# Bucle de entrenamiento optimizado
    for epoch in range(epochs):
        GMF0.train()
        running_loss = 0.0
        
        for batch_u, batch_i, batch_r in loader:
            batch_u = batch_u.to(device, non_blocking=True)
            batch_i = batch_i.to(device, non_blocking=True)
            batch_r = batch_r.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            
            outputs = GMF0(batch_u, batch_i)
            loss = loss_fn(outputs, batch_r)
            
            loss.backward()
            optimizer.step()
            
            # Acumulamos directamente el tensor en la GPU sin llamar a .item() continuamente
            running_loss += loss.detach()

        # Pasamos a CPU y calculamos la media solo UNA vez al terminar la época
        epoch_loss = running_loss.item() / len(loader)
        print(f"Época {epoch+1}/{epochs} - Pérdida (MSE): {epoch_loss:.4f}")

# 3 min

Época 1/10 - Pérdida (MSE): 42.1691
Época 2/10 - Pérdida (MSE): 20.0007
Época 3/10 - Pérdida (MSE): 7.8377
Época 4/10 - Pérdida (MSE): 3.1070
Época 5/10 - Pérdida (MSE): 2.3555
Época 6/10 - Pérdida (MSE): 2.2088
Época 7/10 - Pérdida (MSE): 2.0546
Época 8/10 - Pérdida (MSE): 1.9212
Época 9/10 - Pérdida (MSE): 1.8153
Época 10/10 - Pérdida (MSE): 1.7277


In [51]:
ruta0 = Path("GMF0_wights.pth")
if not ruta0.exists():
    torch.save(GMF0.state_dict(), 'GMF0_wights.pth')

In [53]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

GMF0_recargado = GMFModel(NUM_USERS, NUM_ITEMS, latent_dim=5).to(device)
GMF0_recargado.load_state_dict(torch.load(ruta0, map_location=device))

C:\Users\Usuario\AppData\Local\Temp\ipykernel_11404\1276168509.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  GMF0_recargado.load_state_dict(torch.load(ruta0, map_locat

<All keys matched successfully>

In [58]:
GMF0.eval()
with torch.no_grad():
    # Convertimos los datos de test a tensores y los enviamos a la GPU
    users_test = torch.tensor(X_test['user_id'].values, dtype=torch.long).to(device)
    items_test = torch.tensor(X_test['anime_id'].values, dtype=torch.long).to(device)

    # Devolvemos las predicciones a la CPU para poder usar numpy/sklearn
    y_pred0 = GMF0(users_test, items_test).cpu().numpy()
y_pred0

array([7.308547 , 9.137329 , 7.809985 , ..., 7.560561 , 7.5684276,
       7.394732 ], shape=(1228986,), dtype=float32)

In [ ]:
'''GMF0_recargado.eval()
with torch.no_grad():
    # Convertimos los datos de test a tensores y los enviamos a la GPU
    users_test = torch.tensor(X_test['user_id'].values, dtype=torch.long).to(device)
    items_test = torch.tensor(X_test['anime_id'].values, dtype=torch.long).to(device)

    # Devolvemos las predicciones a la CPU para poder usar numpy/sklearn
    y_pred0 = GMF0_recargado(users_test, items_test).cpu().numpy()
y_pred0'''

array([7.308547 , 9.137329 , 7.809985 , ..., 7.560561 , 7.5684276,
       7.394732 ], shape=(1228986,), dtype=float32)

In [59]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(y_test, y_pred0)

0.9998140931129456

#### Parámetro óptimo

In [ ]:
import optuna
import copy

# 1. Definimos la función objetivo que Optuna va a optimizar
def objective(trial):
    # Definimos los rangos de búsqueda inteligente
    # Optuna elegirá un entero sugerido entre 8 y 64 para la dimensión latente
    d_latente = trial.suggest_int('latent_dim', 8, 64, step=8)
    # También podemos optimizar el learning rate en escala logarítmica
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    
    # Instanciamos el modelo con los parámetros sugeridos por Optuna
    modelo_gmf = GMFModel(NUM_USERS, NUM_ITEMS, d_latente).to(device)
    optimizer = torch.optim.Adam(modelo_gmf.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    
    # Parámetros fijos de control para no eternizar la prueba
    epochs_prueba = 5 
    
    for epoch in range(epochs_prueba):
        modelo_gmf.train()
        for batch_u, batch_i, batch_r in loader:
            batch_u = batch_u.to(device, non_blocking=True)
            batch_i = batch_i.to(device, non_blocking=True)
            batch_r = batch_r.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            outputs = modelo_gmf(batch_u, batch_i)
            loss = loss_fn(outputs, batch_r)
            loss.backward()
            optimizer.step()
            
        # Evaluamos al final de la última época para devolverle el MAE a Optuna
        modelo_gmf.eval()
        with torch.no_grad():
            users_test = torch.tensor(X_test['user_id'].values, dtype=torch.long).to(device)
            items_test = torch.tensor(X_test['anime_id'].values, dtype=torch.long).to(device)
            preds_test = modelo_gmf(users_test, items_test).cpu().numpy()
            
        epoch_test_mae = mean_absolute_error(y_test, preds_test)
        
        # Reportamos el progreso a Optuna para que pueda aplicar "Pruning" 
        # (si ve que la prueba va muy mal comparada con las anteriores, la corta a la mitad)
        trial.report(epoch_test_mae, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
            
    return epoch_test_mae  # Este es el valor que Optuna intentará minimizar

# 2. Creamos el estudio de Optuna enfocado en MINIMIZAR el MAE
study = optuna.create_study(direction='minimize')

# Lanzamos la búsqueda para que haga, por ejemplo, 7 intentos inteligentes
study.optimize(objective, n_trials=7)

# --- RESULTADOS ---
print("\n" + "🏆" * 20)
print("¡BÚSQUEDA COMPLETADA POR OPTUNA!")
print(f"Mejor MAE conseguido: {study.best_value:.4f}")
print(f"Parámetros óptimos encontrados: {study.best_params}")

# 8 minutos

c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-05-16 13:05:01,763] A new study created in memory with name: no-name-8f19e141-64a1-45e1-b068-96501e9a70d2
[I 2026-05-16 13:06:38,595] Trial 0 finished with value: 0.9621820449829102 and parameters: {'latent_dim': 16, 'lr': 0.002530051386916138}. Best is trial 0 with value: 0.9621820449829102.
[I 2026-05-16 13:08:17,708] Trial 1 finished with value: 0.9151417016983032 and parameters: {'latent_dim': 24, 'lr': 0.0032392825979785417}. Best is trial 1 with value: 0.9151417016983032.
[I 2026-05-16 13:09:41,125] Trial 2 finished with value: 0.8981948494911194 and parameters: {'latent_dim': 40, 'lr': 0.006288390485357102}. Best is trial 2 with value: 0.8981948494911194.
[I 2026-05-16 13:11:00,227] Trial 3 finis


🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆
¡BÚSQUEDA COMPLETADA POR OPTUNA!
Mejor MAE conseguido: 0.8982
Parámetros óptimos encontrados: {'latent_dim': 40, 'lr': 0.006288390485357102}


In [ ]:
# 1. Instanciar el modelo con los parámetros óptimos de Optuna
latent_dim_optimo = 40
lr_optimo = 0.00628

GMF_definitivo = GMFModel(NUM_USERS, NUM_ITEMS, latent_dim_optimo).to(device)
optimizer = torch.optim.Adam(GMF_definitivo.parameters(), lr=lr_optimo)
loss_fn = nn.MSELoss()

# Volvemos a usar el DataLoader rápido
loader_gmf = DataLoader(dataset, batch_size=2048, shuffle=True, num_workers=4, pin_memory=True)

# 2. Entrenamos con 10 épocas para dejarlo niquelado
epochs_final = 10
print(f"🚀 Entrenando GMF definitivo con latent_dim={latent_dim_optimo} durante {epochs_final} épocas...")

for epoch in range(epochs_final):
    GMF_definitivo.train()
    running_loss = 0.0
    
    for batch_u, batch_i, batch_r in loader_gmf:
        batch_u = batch_u.to(device, non_blocking=True)
        batch_i = batch_i.to(device, non_blocking=True)
        batch_r = batch_r.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        outputs = GMF_definitivo(batch_u, batch_i)
        loss = loss_fn(outputs, batch_r)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.detach()
        
    epoch_loss = running_loss.item() / len(loader_gmf)
    print(f"Época {epoch+1:02d}/{epochs_final} - Pérdida (MSE): {epoch_loss:.4f}")

# 3. Evaluación final del MAE de test
GMF_definitivo.eval()
with torch.no_grad():
    users_test = torch.tensor(X_test['user_id'].values, dtype=torch.long).to(device)
    items_test = torch.tensor(X_test['anime_id'].values, dtype=torch.long).to(device)
    preds_test = GMF_definitivo(users_test, items_test).cpu().numpy()

from sklearn.metrics import mean_absolute_error
mae_final_gmf = mean_absolute_error(y_test, preds_test)
print(f"\n🏆 ¡Entrenamiento completado! MAE definitivo en Test: {mae_final_gmf:.4f}")


# 3 min

🚀 Entrenando GMF definitivo con latent_dim=40 durante 10 épocas...
Época 01/10 - Pérdida (MSE): 14.8361
Época 02/10 - Pérdida (MSE): 1.6703
Época 03/10 - Pérdida (MSE): 1.5760
Época 04/10 - Pérdida (MSE): 1.4655
Época 05/10 - Pérdida (MSE): 1.2928
Época 06/10 - Pérdida (MSE): 1.1471
Época 07/10 - Pérdida (MSE): 1.0352
Época 08/10 - Pérdida (MSE): 0.9555
Época 09/10 - Pérdida (MSE): 0.8986
Época 10/10 - Pérdida (MSE): 0.8572

🏆 ¡Entrenamiento completado! MAE definitivo en Test: 0.9081


### Multilayer Perceptron

In [24]:
latent_dim = 16
epochs = 5

In [18]:
class MLPModel(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.item_embedding = nn.Embedding(num_items, latent_dim)
        self.mlp = nn.Sequential(
            nn.Linear(latent_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, user_ids, item_ids):
        u_emb = self.user_embedding(user_ids)
        i_emb = self.item_embedding(item_ids)
        vector = torch.cat([u_emb, i_emb], dim=-1)
        return self.mlp(vector).flatten()

MLP = MLPModel(NUM_USERS, NUM_ITEMS, latent_dim)
print(MLP)

MLPModel(
  (user_embedding): Embedding(47143, 16)
  (item_embedding): Embedding(6532, 16)
  (mlp): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [19]:
optimizer_mlp = torch.optim.Adam(MLP.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

users_t = torch.tensor(X_train['user_id'].values, dtype=torch.long)
items_t = torch.tensor(X_train['anime_id'].values, dtype=torch.long)
ratings_t = torch.tensor(y_train.values, dtype=torch.float32)

dataset = TensorDataset(users_t, items_t, ratings_t)
# Entrenamos el MLP
MLP = MLP.to(device)

# 2. Configurar el DataLoader de alta velocidad (mismo truco de antes)
loader_mlp = DataLoader(
    dataset, 
    batch_size=2048, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True
)

# 3. Bucle de entrenamiento del MLP
print("Iniciando entrenamiento del MLP...")
for epoch in range(epochs): # Usará las epochs que definiste para el MLP (5 épocas)
    MLP.train()
    running_loss = 0.0
    
    for b_u, b_i, b_r in loader_mlp:
        b_u = b_u.to(device, non_blocking=True)
        b_i = b_i.to(device, non_blocking=True)
        b_r = b_r.to(device, non_blocking=True)
        
        optimizer_mlp.zero_grad()
        
        preds = MLP(b_u, b_i)
        loss = loss_fn(preds, b_r)
        
        loss.backward()
        optimizer_mlp.step()
        
        running_loss += loss.detach()
        
    epoch_loss = running_loss.item() / len(loader_mlp)
    print(f"Época {epoch+1} - Pérdida MLP (MSE): {epoch_loss:.4f}")

Iniciando entrenamiento del MLP...
Época 1 - Pérdida MLP (MSE): 3.2956
Época 2 - Pérdida MLP (MSE): 1.6365
Época 3 - Pérdida MLP (MSE): 1.4819
Época 4 - Pérdida MLP (MSE): 1.4387
Época 5 - Pérdida MLP (MSE): 1.4201


In [20]:
MLP.eval()
with torch.no_grad():
    # Preparamos los tensores de test en la GPU
    users_test = torch.tensor(X_test['user_id'].values, dtype=torch.long).to(device)
    items_test = torch.tensor(X_test['anime_id'].values, dtype=torch.long).to(device)
    
    # Predecimos con el MLP y pasamos el resultado a la CPU
    y_pred_mlp = MLP(users_test, items_test).cpu().numpy()
y_pred_mlp

array([7.1019244, 9.18641  , 7.695882 , ..., 6.9459724, 8.196757 ,
       6.834785 ], shape=(1228986,), dtype=float32)

In [21]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(y_test, y_pred_mlp)

0.9183365106582642

#### Valores óptimos


In [ ]:
import optuna
import copy
from sklearn.metrics import mean_absolute_error

# 1. Definimos la función objetivo adaptada EXACTAMENTE a tu clase MLPModel
def objective_mlp(trial):
    # Optuna elegirá la dimensión latente idónea entre 16 y 64
    d_latente = trial.suggest_int('latent_dim', 16, 64, step=16)
    # Optuna elegirá el ritmo de aprendizaje ideal en escala logarítmica
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    
    # Instanciamos tu modelo pasando los parámetros dinámicos
    modelo_mlp = MLPModel(NUM_USERS, NUM_ITEMS, d_latente).to(device)
    
    optimizer = torch.optim.Adam(modelo_mlp.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    
    # DataLoader acelerado para tu RTX 3060 Ti
    loader_mlp_opt = DataLoader(
        dataset, 
        batch_size=2048, 
        shuffle=True, 
        num_workers=4, 
        pin_memory=True
    )
    
    epochs_prueba = 4  # Límite de épocas por intento para no eternizar la búsqueda
    
    for epoch in range(epochs_prueba):
        modelo_mlp.train()
        for batch_u, batch_i, batch_r in loader_mlp_opt:
            batch_u = batch_u.to(device, non_blocking=True)
            batch_i = batch_i.to(device, non_blocking=True)
            batch_r = batch_r.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            preds = modelo_mlp(batch_u, batch_i)
            loss = loss_fn(preds, batch_r)
            loss.backward()
            optimizer.step()
            
        # Evaluación en Test al cerrar cada época
        modelo_mlp.eval()
        with torch.no_grad():
            users_test = torch.tensor(X_test['user_id'].values, dtype=torch.long).to(device)
            items_test = torch.tensor(X_test['anime_id'].values, dtype=torch.long).to(device)
            preds_test = modelo_mlp(users_test, items_test).cpu().numpy()
            
        epoch_test_mae = mean_absolute_error(y_test, preds_test)
        
        # Reportar a Optuna para activar la poda inteligente (Pruning)
        trial.report(epoch_test_mae, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
            
    return epoch_test_mae

# 2. Creamos y lanzamos el estudio de optimización
study_mlp = optuna.create_study(direction='minimize')
study_mlp.optimize(objective_mlp, n_trials=40)

# --- PANEL DE RESULTADOS ---
print("\n" + "👑" * 20)
print("¡BÚSQUEDA DEL MLP COMPLETADA POR OPTUNA!")
print(f"Mejor MAE conseguido en MLP: {study_mlp.best_value:.4f}")
print(f"Parámetros óptimos encontrados: {study_mlp.best_params}")

# 34 min

[I 2026-05-16 13:52:52,034] A new study created in memory with name: no-name-85043d8b-7b1c-41f1-a38c-b132868c53a1
[I 2026-05-16 13:54:03,574] Trial 0 finished with value: 0.93551105260849 and parameters: {'latent_dim': 16, 'lr': 0.0006005771990529946}. Best is trial 0 with value: 0.93551105260849.
[I 2026-05-16 13:55:09,008] Trial 1 finished with value: 1.0252524614334106 and parameters: {'latent_dim': 48, 'lr': 0.00015101632455179632}. Best is trial 0 with value: 0.93551105260849.
[I 2026-05-16 13:56:13,050] Trial 2 finished with value: 0.9122046828269958 and parameters: {'latent_dim': 64, 'lr': 0.0056704570200692885}. Best is trial 2 with value: 0.9122046828269958.
[I 2026-05-16 13:57:18,294] Trial 3 finished with value: 0.9207803010940552 and parameters: {'latent_dim': 48, 'lr': 0.0014542490344035966}. Best is trial 2 with value: 0.9122046828269958.
[I 2026-05-16 13:58:22,183] Trial 4 finished with value: 0.9308857917785645 and parameters: {'latent_dim': 32, 'lr': 0.0005297032744279


👑👑👑👑👑👑👑👑👑👑👑👑👑👑👑👑👑👑👑👑
¡BÚSQUEDA DEL MLP COMPLETADA POR OPTUNA!
Mejor MAE conseguido en MLP: 0.9122
Parámetros óptimos encontrados: {'latent_dim': 64, 'lr': 0.0056704570200692885}


In [29]:
# 1. Instanciamos el MLP definitivo con el veredicto de Optuna
latent_dim_campeon = 64
lr_campeon = 0.00567

MLP_definitivo = MLPModel(NUM_USERS, NUM_ITEMS, latent_dim_campeon).to(device)
optimizer_mlp = torch.optim.Adam(MLP_definitivo.parameters(), lr=lr_campeon)
loss_fn = nn.MSELoss()

# DataLoader de alta velocidad (mismo truco para exprimir tu GPU)
loader_mlp_final = DataLoader(
    dataset, 
    batch_size=2048, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True
)

# 2. Bucle de entrenamiento final estirado a 10 épocas
epochs_final = 10
print(f"🚀 Entrenando MLP definitivo con latent_dim={latent_dim_campeon} durante {epochs_final} épocas...")

for epoch in range(epochs_final):
    MLP_definitivo.train()
    running_loss = 0.0
    
    for b_u, b_i, b_r in loader_mlp_final:
        b_u = b_u.to(device, non_blocking=True)
        b_i = b_i.to(device, non_blocking=True)
        b_r = b_r.to(device, non_blocking=True)
        
        optimizer_mlp.zero_grad()
        
        preds = MLP_definitivo(b_u, b_i)
        loss = loss_fn(preds, b_r)
        
        loss.backward()
        optimizer_mlp.step()
        
        running_loss += loss.detach()
        
    epoch_loss = running_loss.item() / len(loader_mlp_final)
    print(f"Época {epoch+1:02d}/{epochs_final} - Pérdida MLP (MSE): {epoch_loss:.4f}")

# 3. Evaluación definitiva del MAE en el conjunto de Test
MLP_definitivo.eval()
with torch.no_grad():
    users_test = torch.tensor(X_test['user_id'].values, dtype=torch.long).to(device)
    items_test = torch.tensor(X_test['anime_id'].values, dtype=torch.long).to(device)
    y_pred_mlp_final = MLP_definitivo(users_test, items_test).cpu().numpy()

from sklearn.metrics import mean_absolute_error
mae_final_mlp = mean_absolute_error(y_test, y_pred_mlp_final)
print(f"\n🏆 ¡Gran Final Completada! MAE definitivo del MLP en Test: {mae_final_mlp:.4f}")

🚀 Entrenando MLP definitivo con latent_dim=64 durante 10 épocas...
Época 01/10 - Pérdida MLP (MSE): 1.8806
Época 02/10 - Pérdida MLP (MSE): 1.4603
Época 03/10 - Pérdida MLP (MSE): 1.4031
Época 04/10 - Pérdida MLP (MSE): 1.3596
Época 05/10 - Pérdida MLP (MSE): 1.3172
Época 06/10 - Pérdida MLP (MSE): 1.2745
Época 07/10 - Pérdida MLP (MSE): 1.2303
Época 08/10 - Pérdida MLP (MSE): 1.1839
Época 09/10 - Pérdida MLP (MSE): 1.1402
Época 10/10 - Pérdida MLP (MSE): 1.0975

🏆 ¡Gran Final Completada! MAE definitivo del MLP en Test: 0.9305
